## Lab 1 - Part B : Forward Kinematics with Robotics Toolbox

In [ ]:
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot, DHRobot, RevoluteMDH, PrismaticMDH
from spatialmath import SE3
import spatialgeometry as sg
import swift
import math

HERE = Path.cwd().as_posix()
links, name, _, _ = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)
urdf_robot = Robot(links, name=name)
print('loaded', name, '|', urdf_robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True)
env.add(urdf_robot)

print(urdf_robot.fkine(np.zeros(urdf_robot.n)))         # forward kinematics of the robot at zero joint angles

### 1 - Check hand calculation with library

In [ ]:
M = np.array([0.0, 0.0, 0.0])                       # edit here (x,y,z) in m
q_handcalc = np.array([0.0, 0.0, 0.0])              # edit here (rad or m)

if np.allclose(M, 0) or np.allclose(q_handcalc, 0):
    print('Please fill in M and q_handcalc with your own numbers, then run this cell again.')
else:
    marker = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6), collision=False)
    marker.T = SE3(*M)
    env.add(marker)

    urdf_robot.q = q_handcalc
    env.step()

    p = urdf_robot.fkine(q_handcalc).t
    print('tip   :', np.round(p, 4))
    print('target:', np.round(M, 4))
    print('error :', round(float(np.linalg.norm(p - M)), 4), 'm')


### 2 - Build your model and compute FK

In [ ]:
# Allowable syntax (reference only - do not run as-is):
# DHRobot([link, link, ...], name='...')                  # build a robot from a list of links
# RevoluteMDH(a=..., alpha=..., d=..., offset=...)        # revolute joint; joint variable is the angle
# PrismaticMDH(theta=..., a=..., alpha=..., offset=...)   # prismatic joint; joint variable is the distance
# robot.fkine(q)                                          # returns a transform T; tip position is T.t

In [ ]:
# (a) Build YOUR robot
robot_mdh = DHRobot([
    RevoluteMDH(a=0.0,  alpha=0.0,   d=0.0),
    RevoluteMDH(a=0.0,  alpha=0.0,  d=0.0),
    RevoluteMDH(a=0.0, alpha=0.0,   d=0.0),
], name='my_mdh')

robot_mdh.tool = SE3.Tx(0.0)           # edit here: set the tool transform to match your robot's end-effector

print(robot_mdh)

In [ ]:
q_A = np.array([np.deg2rad(30), np.deg2rad(30), np.deg2rad(30)])  # edit here (prismatic_max * 0.6 for prismatic joints)
q_B = q_handcalc

# edit here to compute T, tip, and dist for q_A and q_B
for label, q in [("q_A", q_A), ("q_B", q_B)]:
    T = None              # (1) forward kinematics -> transform
    tip = None            # (2) tip position (x, y, z)
    dist = None           # (3) distance from tip to target M
    print(label, "tip:", np.round(tip, 4))
    print("dist:", round(float(dist), 4), "m") if label == 'q_B' else print("\n")

### 3 - Cross-check your MDH model vs your URDF model

In [ ]:
q = [q_A, q_B]
for joint in q:
    print('Joint Test: ', joint)
    print('MDH  tip:', np.round(robot_mdh.fkine(joint).t, 4))
    print('URDF tip:', np.round(urdf_robot.fkine(joint).t, 4))
    print('diff   :', round(float(np.linalg.norm(robot_mdh.fkine(joint).t - urdf_robot.fkine(joint).t)), 4), 'm')
    print('----------------------------------- \n')